[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Worked Designs &middot; Solutions


One way to build the bank account. Not the only way. If yours runs and does what was asked, yours
is right too, and a design that differs from this one may well be better.

Run the cell below first.


In [1]:
from dataclasses import dataclass, field

print("ready")


ready


**1.** A transaction is a value, so it is a frozen dataclass.


In [2]:
@dataclass(frozen=True)
class Transaction:
    amount: float
    description: str

    def __post_init__(self):
        if self.amount == 0:
            raise ValueError("a transaction cannot be for nothing")


deposit = Transaction(100.0, "salary")
withdrawal = Transaction(-35.5, "groceries")
print(deposit)
print(withdrawal)

try:
    Transaction(0, "nothing")
except ValueError as error:
    print("refused:", error)


Transaction(amount=100.0, description='salary')
Transaction(amount=-35.5, description='groceries')
refused: a transaction cannot be for nothing


A transaction never changes once it has happened, which is exactly what `frozen=True` expresses. The
sign carries the direction: positive money in, negative money out. That choice makes the balance a
simple sum.


**2.** An account holds transactions, and works its balance out from them.


In [3]:
@dataclass
class Account:
    owner: str
    transactions: list[Transaction] = field(default_factory=list, repr=False)

    @property
    def balance(self):
        return round(sum(t.amount for t in self.transactions), 2)

    def deposit(self, amount, description):
        if amount <= 0:
            raise ValueError("a deposit must be more than zero")
        self.transactions.append(Transaction(amount, description))

    def withdraw(self, amount, description):
        if amount <= 0:
            raise ValueError("a withdrawal must be more than zero")
        self.transactions.append(Transaction(-amount, description))


ada = Account("Ada")
ada.deposit(100.0, "salary")
ada.withdraw(35.5, "groceries")
print("balance:", ada.balance)

try:
    ada.balance = 1_000_000
except AttributeError as error:
    print("refused:", error)


balance: 64.5
refused: property 'balance' of 'Account' object has no setter


The balance is never stored, so it can never disagree with the transactions. It is a property with no
setter, so nothing can assign a balance directly: the only way to change it is to record a transaction,
which leaves a history behind.


**3.** An exception family, and a withdrawal that checks.


In [4]:
class AccountError(Exception):
    """Anything that goes wrong with an account."""


class InsufficientFundsError(AccountError):
    def __init__(self, balance, requested):
        super().__init__(f"cannot withdraw {requested:.2f}: the balance is {balance:.2f}")
        self.balance = balance
        self.requested = requested


@dataclass
class Account:
    owner: str
    transactions: list[Transaction] = field(default_factory=list, repr=False)

    @property
    def balance(self):
        return round(sum(t.amount for t in self.transactions), 2)

    def deposit(self, amount, description):
        if amount <= 0:
            raise ValueError("a deposit must be more than zero")
        self.transactions.append(Transaction(amount, description))

    def withdraw(self, amount, description):
        if amount <= 0:
            raise ValueError("a withdrawal must be more than zero")
        if amount > self.balance:
            raise InsufficientFundsError(self.balance, amount)
        self.transactions.append(Transaction(-amount, description))


ada = Account("Ada")
ada.deposit(100.0, "salary")

try:
    ada.withdraw(500, "a bicycle")
except InsufficientFundsError as error:
    print("refused:", error)
    print("missing:", round(error.requested - error.balance, 2))


refused: cannot withdraw 500.00: the balance is 100.00
missing: 400.0


The handler worked out what was missing from the exception's attributes, not from its message. A
caller who handles every account problem the same way can catch `AccountError` instead.


**4.** Length, looping, and a readable printout.


In [5]:
@dataclass
class Account:
    owner: str
    transactions: list[Transaction] = field(default_factory=list, repr=False)

    @property
    def balance(self):
        return round(sum(t.amount for t in self.transactions), 2)

    def deposit(self, amount, description):
        if amount <= 0:
            raise ValueError("a deposit must be more than zero")
        self.transactions.append(Transaction(amount, description))

    def withdraw(self, amount, description):
        if amount <= 0:
            raise ValueError("a withdrawal must be more than zero")
        if amount > self.balance:
            raise InsufficientFundsError(self.balance, amount)
        self.transactions.append(Transaction(-amount, description))

    def __len__(self):
        return len(self.transactions)

    def __iter__(self):
        return iter(self.transactions)

    def __repr__(self):
        return f"{type(self).__name__}({self.owner!r}, balance={self.balance:.2f})"


ada = Account("Ada")
ada.deposit(100.0, "salary")
ada.withdraw(35.5, "groceries")

print(ada, "with", len(ada), "transactions:")
for transaction in ada:
    print(" ", transaction)


Account('Ada', balance=64.50) with 2 transactions:
  Transaction(amount=100.0, description='salary')
  Transaction(amount=-35.5, description='groceries')


A dataclass keeps a `__repr__` you write yourself instead of generating one. Writing
`type(self).__name__` rather than `Account` matters in task 6, where a savings account should print as
one.


**5.** An alternative constructor for a bank statement.


In [6]:
class Account(Account):
    @classmethod
    def from_statement(cls, owner, lines):
        account = cls(owner)
        for line in lines:
            amount, description = line.split(" ", 1)
            account.transactions.append(Transaction(float(amount), description))
        return account


grace = Account.from_statement("Grace", ["+250.00 salary", "-12.40 lunch", "-60.00 train pass"])

print(grace)
print([t.description for t in grace])


Account('Grace', balance=177.60)
['salary', 'lunch', 'train pass']


`split(" ", 1)` splits at the first space only, so a description with spaces in it, such as
`train pass`, stays whole. `float("+250.00")` accepts the sign, which is why the statement format can
show deposits with a `+`.

`class Account(Account):` builds a new `Account` on top of the one from task 4, adding one method,
so the earlier code did not have to be repeated. In a file, you would add `from_statement` to the one
class instead.


**6.** A savings account, as a kind of account.


In [7]:
class TooManyWithdrawalsError(AccountError):
    def __init__(self, limit):
        super().__init__(f"a savings account allows {limit} withdrawals")
        self.limit = limit


class SavingsAccount(Account):
    WITHDRAWAL_LIMIT = 2

    def withdraw(self, amount, description):
        made = sum(1 for t in self.transactions if t.amount < 0)
        if made >= self.WITHDRAWAL_LIMIT:
            raise TooManyWithdrawalsError(self.WITHDRAWAL_LIMIT)
        super().withdraw(amount, description)

    def add_interest(self, rate):
        self.deposit(round(self.balance * rate, 2), f"interest at {rate:.1%}")


savings = SavingsAccount("Ada")
savings.deposit(1000, "opening deposit")
savings.add_interest(0.02)
savings.withdraw(50, "a gift")
savings.withdraw(20, "a book")

try:
    savings.withdraw(5, "coffee")
except TooManyWithdrawalsError as error:
    print("refused:", error)

print(savings)
print("is an Account:", isinstance(savings, Account))


refused: a savings account allows 2 withdrawals
SavingsAccount('Ada', balance=950.00)
is an Account: True


`SavingsAccount` changes one rule and adds one ability, and inherits everything else. Its `withdraw`
checks the new limit and hands the rest to `super().withdraw`, so the insufficient-funds check still
applies. And it prints as `SavingsAccount`, because task 4 wrote `type(self).__name__`.

A savings account is genuinely a kind of account, and every `Account` method makes sense on it, which
is the test from the **Composition over Inheritance** notebook for when inheriting is right.


---

&#8592; **Back to:** [Worked Designs](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/14-worked-designs.ipynb)  &nbsp;&middot;&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)
